# Which grouping should we keep?

**ITCA measures prediction support; ITCA+ asks whether extra groups justify their complexity.** We compare three fixed synthetic groupings. Molecular expression distinguishes M0–M3; Q0–Q2 share one program. These teaching labels are defined before evaluation.

New to ITCA? [Start with the concept](00_concepts.ipynb).

In [ ]:
from IPython.display import display
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gauge_sc as gauge

adata = gauge.datasets.audit_example()
group = adata.obs["group"].astype(str)
adata.obs["coarse"] = group.map({
    "M0": "M01", "M1": "M01", "M2": "M23", "M3": "M23",
    "Q0": "Q", "Q1": "Q", "Q2": "Q",
})
adata.obs["supported"] = group.where(~group.str.startswith("Q"), "Q")
adata.obs["fine"] = group
result = gauge.evaluate(
    adata, groupby=["coarse", "supported", "fine"],
    views={"molecular": {"obsm": "X_pca"}}, aggregation="fold_mean",
)

## Inspect the trade-off

Each additional group costs `−p log(p)` nats. `p` calibrates the penalty; it does not remove small groups. Choose it explicitly.

In [ ]:
selection = gauge.select(result, view="molecular", population_ratio=0.01)
display(selection.table.round(3))


## Explore the same candidates without retraining

Larger penalties favor fewer groups. Points below show the requested parameter grid, not exact transition locations. Selection is conditional on these candidates and the molecular view.

In [ ]:
selection = gauge.select(
    result, view="molecular", population_ratio=np.linspace(0.001, 0.3, 20)
)
with sns.plotting_context("paper"), plt.rc_context({
    "text.color": "black", "axes.labelcolor": "black", "axes.edgecolor": "black",
    "xtick.color": "black", "ytick.color": "black", "axes.grid": False,
}):
    fig, ax = plt.subplots(figsize=(4.8, 3.2), layout="constrained")
    ax.scatter(selection.path.population_ratio, selection.path.n_groups, color="black", s=22)
    ax.set(xlabel="Population ratio p", ylabel="Selected number of groups", yticks=[3, 5, 7])
    sns.despine(ax=ax)
    plt.show()
selection.path[["population_ratio", "groupby", "n_groups", "itca", "penalty", "itca_plus"]].round(3)

For your data, supply existing clustering columns on the same AnnData and an explicit molecular representation. Candidate generation is external; fixed-label evaluation does not independently validate the entire clustering workflow. [API details](../docs/api.md#select) · [Audit another view](01_view_specific_audit.ipynb)

## Work with your agent

[Install the skill and choose an interface](../docs/agents.md). Then try:

> Use $gauge to compare the supplied candidate groupings on a common molecular view. Explain the tradeoff between ITCA and retained group count. Recommend a penalty range appropriate to my goal, or ask about the desired population scale, and show the resulting selection path.

These tutorial settings are an example; adapt them to your question and data.